# Notebook 5 — Synthetic Shopping Query Generation

## Objective

This notebook uses the Qwen2.5-7B-Instruct language model to generate
realistic synthetic shopping queries using both user preference profiles
and held-out target products.

The user preference profile represents the user's historical purchasing
behaviour, while the held-out target product represents the future item
associated with the recommendation instance.

The target product is used during synthetic query construction so that the
generated query corresponds to the evaluation target. However, the model
is instructed not to reveal the target product's exact identity, including
its exact title, ASIN, or unique model identifier.

The workflow consists of:

1. Loading processed user preference profiles.
2. Loading user-history instances containing held-out ground-truth products.
3. Linking each preference profile to its corresponding held-out target.
4. Loading the Qwen2.5-7B-Instruct model.
5. Defining a target-aware query generation prompt.
6. Testing query generation on individual profile-target pairs.
7. Generating synthetic queries for the complete dataset.
8. Saving checkpoints to support recovery from interrupted GPU jobs.
9. Performing profile-grounding, target-alignment, and leakage checks.
10. Creating the final synthetic-query dataset.

## 1. Environment and Project Configuration

The project uses the processed user preference profiles and user interaction
histories generated in previous notebooks.

User preference profiles represent only historical information. The
held-out ground-truth product is kept separate and is used to construct
the synthetic query without being incorporated into the user's preference
profile.

The project directory is explicitly specified to ensure that file paths
remain consistent across different SLURM nodes and Jupyter sessions.

In [3]:
import os
os.environ["HTTP_PROXY"] = "http://hpc-proxy00.city.ac.uk:3128"
os.environ["HTTPS_PROXY"] = "http://hpc-proxy00.city.ac.uk:3128"
os.environ["http_proxy"] = "http://hpc-proxy00.city.ac.uk:3128"
os.environ["https_proxy"] = "http://hpc-proxy00.city.ac.uk:3128"
print("Proxy configured")

Proxy configured


In [4]:
from pathlib import Path
import json
import re
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
tqdm.pandas()

/users/adhp511/.pyenv/versions/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
PROJECT_ROOT= Path.home()/"query-conditioned-recommender"
DATA_DIR= PROJECT_ROOT/"data"
PROCESSED_DIR= DATA_DIR/"processed"
GENERATED_DIR= DATA_DIR/"generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME= "Qwen/Qwen2.5-7B-Instruct"
DEVICE= "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")

Running on: cuda


In [6]:
# Load user preference profiles
profiles = pd.read_parquet(
    PROCESSED_DIR / "user_preference_profiles.parquet"
)

# Load user-history instances containing the held-out target
histories = pd.read_parquet(
    PROCESSED_DIR / "user_histories.parquet"
)

print("Profiles shape:", profiles.shape)
print("Histories shape:", histories.shape)

print("\nProfile columns:")
print(profiles.columns.tolist())

print("\nHistory columns:")
print(histories.columns.tolist())

Profiles shape: (5559, 26)
Histories shape: (5559, 12)

Profile columns:
['sample_id', 'user_id', 'history_asins', 'ground_truth_asin', 'history_length', 'history_timestamps', 'history_ratings', 'ground_truth_timestamp', 'ground_truth_rating', 'query', 'constraint_type', 'constraint_values', 'history_titles', 'history_categories', 'history_descriptions', 'ground_truth_title', 'ground_truth_categories', 'ground_truth_description', 'history_main_categories', 'preferred_categories', 'average_rating', 'liked_products', 'recent_products', 'category_diversity', 'purchase_span_days', 'profile_text']

History columns:
['sample_id', 'user_id', 'history_asins', 'ground_truth_asin', 'history_length', 'history_timestamps', 'history_ratings', 'ground_truth_timestamp', 'ground_truth_rating', 'query', 'constraint_type', 'constraint_values']


In [7]:
# Select only the information needed to associate each user profile
# with its held-out ground-truth product.

target_data = histories[
    [
        "sample_id",
        "user_id",
        "ground_truth_asin",
        "ground_truth_timestamp",
        "ground_truth_rating"
    ]
].copy()

print("Target data shape:", target_data.shape)

print("\nTarget data columns:")
print(target_data.columns.tolist())

display(target_data.head())

Target data shape: (5559, 5)

Target data columns:
['sample_id', 'user_id', 'ground_truth_asin', 'ground_truth_timestamp', 'ground_truth_rating']


,sample_id,user_id,ground_truth_asin,ground_truth_timestamp,ground_truth_rating
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B007VKII6A,2017-05-21 14:36:52.000,2
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B07454F4JH,2017-08-24 17:49:29.844,5
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00SMVLPIK,2017-08-24 17:57:36.638,5
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B0179DDIQI,2017-08-24 18:06:02.421,5
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00X088DZI,2019-05-11 17:22:30.699,5


In [8]:
# Step 3 — Use the existing target information in the profile dataset

# Reload the profile dataset cleanly
profiles = pd.read_parquet(
    PROCESSED_DIR / "user_preference_profiles.parquet"
)

# The profile dataset already contains:
# ground_truth_asin
# ground_truth_timestamp
# ground_truth_rating
# Therefore, no merge with user_histories is required.

dataset = profiles.copy()

print("Dataset shape:", dataset.shape)

print("\nRelevant columns:")
print(
    dataset[
        [
            "sample_id",
            "user_id",
            "ground_truth_asin",
            "ground_truth_timestamp",
            "ground_truth_rating",
            "profile_text"
        ]
    ].head()
)

print("\nGround-truth ASIN missing:", dataset["ground_truth_asin"].isna().sum())
print("Unique sample IDs:", dataset["sample_id"].nunique())

Dataset shape: (5559, 26)

Relevant columns:
   sample_id                       user_id ground_truth_asin  \
0          0  AE2MIK6YBTBRNZZGI5O5Z6VCMEPA        B007VKII6A   
1          1  AE2MIK6YBTBRNZZGI5O5Z6VCMEPA        B07454F4JH   
2          2  AE2MIK6YBTBRNZZGI5O5Z6VCMEPA        B00SMVLPIK   
3          3  AE2MIK6YBTBRNZZGI5O5Z6VCMEPA        B0179DDIQI   
4          4  AE2MIK6YBTBRNZZGI5O5Z6VCMEPA        B00X088DZI   

   ground_truth_timestamp  ground_truth_rating  \
0 2017-05-21 14:36:52.000                    2   
1 2017-08-24 17:49:29.844                    5   
2 2017-08-24 17:57:36.638                    5   
3 2017-08-24 18:06:02.421                    5   
4 2019-05-11 17:22:30.699                    5   

                                        profile_text  
0  User Preference Profile\n\nThe user has primar...  
1  User Preference Profile\n\nThe user has primar...  
2  User Preference Profile\n\nThe user has primar...  
3  User Preference Profile\n\nThe user has primar

In [9]:
# Step 4 — Attach target product metadata using the held-out ASIN

candidate_metadata = pd.read_parquet(
    GENERATED_DIR / "candidate_metadata.parquet"
)

print("Candidate metadata shape:", candidate_metadata.shape)
print("Candidate metadata columns:")
print(candidate_metadata.columns.tolist())

# Keep the metadata fields we need for target-aware query generation
target_metadata = candidate_metadata[
    [
        "parent_asin",
        "main_category",
        "title",
        "features",
        "description",
        "price",
        "categories"
    ]
].copy()

# Rename the ASIN column so the merge key is explicit
target_metadata = target_metadata.rename(
    columns={"parent_asin": "ground_truth_asin"}
)

# Make sure the merge key has the same type
dataset["ground_truth_asin"] = dataset["ground_truth_asin"].astype(str)
target_metadata["ground_truth_asin"] = target_metadata["ground_truth_asin"].astype(str)

# Attach target metadata
dataset = dataset.merge(
    target_metadata,
    on="ground_truth_asin",
    how="left",
    validate="many_to_one"
)

print("\nDataset shape after target metadata:", dataset.shape)

print("\nTarget metadata preview:")
print(
    dataset[
        [
            "sample_id",
            "ground_truth_asin",
            "main_category",
            "title",
            "price"
        ]
    ].head()
)

print(
    "\nTargets without metadata:",
    dataset["title"].isna().sum()
)

Candidate metadata shape: (6585, 21)
Candidate metadata columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author', 'title_text', 'category_text', 'feature_text', 'description_text', 'product_text']

Dataset shape after target metadata: (5559, 32)

Target metadata preview:
   sample_id ground_truth_asin         main_category  \
0          0        B007VKII6A   Musical Instruments   
1          1        B07454F4JH             Computers   
2          2        B00SMVLPIK  Home Audio & Theater   
3          3        B0179DDIQI       All Electronics   
4          4        B00X088DZI       Office Products   

                                               title   price  
0  IDANCE CRAZY101 - Watt - Channel Recording Stu...          
1  Seagate Backup Plus Portable 5TB External Hard...  279.99  
2  Panasonic FOLDZ On-Ear Stereo Headphones

In [10]:
# Step 5 — Construct leakage-safe target information for Qwen
#
# We deliberately do NOT pass the target product title or ASIN to Qwen.
# The target metadata is used only to describe the general product
# characteristics needed for synthetic query generation.

def clean_metadata_value(value):
    if pd.isna(value):
        return ""
    
    if isinstance(value, list):
        return " ".join(str(x) for x in value)
    
    return str(value)


def build_target_info(row):
    parts = []

    # Product category
    if row["main_category"]:
        parts.append(f"Category: {clean_metadata_value(row['main_category'])}")

    # Product features
    features = clean_metadata_value(row["features"])
    if features:
        parts.append(f"Features: {features}")

    # Product description
    description = clean_metadata_value(row["description"])
    if description:
        parts.append(f"Description: {description}")

    # Additional category information
    categories = clean_metadata_value(row["categories"])
    if categories:
        parts.append(f"Product categories: {categories}")

    return "\n".join(parts)


dataset["target_product_info"] = dataset.apply(
    build_target_info,
    axis=1
)

print("Target information created.")

print("\nExample target information:")
print(dataset.loc[0, "target_product_info"])

print("\nMissing target information:",
      (dataset["target_product_info"].str.strip() == "").sum())

print("\nAverage target information length:",
      dataset["target_product_info"].str.len().mean())

Target information created.

Example target information:
Category: Musical Instruments
Features: ['Super Aticulated Headband,Extra Bass with Open Ear CUP,Foldable for EZ transport,Stereo headphone for iphone,with inline microphone,Light weight for outdoor digital audio,Customized Comfortable soft ear pad and headband', 'Driver Unit: 44mm, Driver Type: Neodymium, Impedance: 32, Sensitivity: 108dB+/-3dB', 'Frequency Response: 15Hz~20KHz, Cord Length: Approx.1.5m, Cable Type: PVC, Plug: 3.5mm ,gold chrome']
Description: ['Super Aticulated Headband. Extra Bass with Open Ear CUP. Foldable for EZ transport, Stereo headphone for iphone. With inline microphone,Light weight for outdoor digital audio, Customized Comfortable soft ear pad and headband, Driver Unit: 44mm, Driver Type: Neodymium, Impedance: 32, Sensitivity: 108dB+/-3dB, Frequency Response: 15Hz~20KHz. Cord Length: Approx.1.5m. Cable Type: PVC. Plug: 3.5mm. Gold Chrome']
Product categories: ['Electronics', 'Headphones, Earbuds & Acce

In [11]:
# Inspect several target-information examples for possible target leakage

for i in range(5):
    row = dataset.iloc[i]

    print("=" * 100)
    print(f"Sample ID: {row['sample_id']}")
    print(f"Target ASIN: {row['ground_truth_asin']}")
    print(f"Target title (REFERENCE ONLY): {row['title']}")
    print("-" * 100)
    print(row["target_product_info"])
    print()

Sample ID: 0
Target ASIN: B007VKII6A
Target title (REFERENCE ONLY): IDANCE CRAZY101 - Watt - Channel Recording Studio Equipment, White and Pink
----------------------------------------------------------------------------------------------------
Category: Musical Instruments
Features: ['Super Aticulated Headband,Extra Bass with Open Ear CUP,Foldable for EZ transport,Stereo headphone for iphone,with inline microphone,Light weight for outdoor digital audio,Customized Comfortable soft ear pad and headband', 'Driver Unit: 44mm, Driver Type: Neodymium, Impedance: 32, Sensitivity: 108dB+/-3dB', 'Frequency Response: 15Hz~20KHz, Cord Length: Approx.1.5m, Cable Type: PVC, Plug: 3.5mm ,gold chrome']
Description: ['Super Aticulated Headband. Extra Bass with Open Ear CUP. Foldable for EZ transport, Stereo headphone for iphone. With inline microphone,Light weight for outdoor digital audio, Customized Comfortable soft ear pad and headband, Driver Unit: 44mm, Driver Type: Neodymium, Impedance: 32, Sen

In [12]:
for i in range(5):
    row = dataset.iloc[i]

    print("=" * 100)
    print(f"Sample ID: {row['sample_id']}")
    print(f"Target title (REFERENCE ONLY): {row['title']}")
    print("-" * 100)
    print("CATEGORY:")
    print(row["main_category"])
    print("\nFEATURES:")
    print(clean_metadata_value(row["features"]))
    print()

Sample ID: 0
Target title (REFERENCE ONLY): IDANCE CRAZY101 - Watt - Channel Recording Studio Equipment, White and Pink
----------------------------------------------------------------------------------------------------
CATEGORY:
Musical Instruments

FEATURES:
['Super Aticulated Headband,Extra Bass with Open Ear CUP,Foldable for EZ transport,Stereo headphone for iphone,with inline microphone,Light weight for outdoor digital audio,Customized Comfortable soft ear pad and headband', 'Driver Unit: 44mm, Driver Type: Neodymium, Impedance: 32, Sensitivity: 108dB+/-3dB', 'Frequency Response: 15Hz~20KHz, Cord Length: Approx.1.5m, Cable Type: PVC, Plug: 3.5mm ,gold chrome']

Sample ID: 1
Target title (REFERENCE ONLY): Seagate Backup Plus Portable 5TB External Hard Drive HDD – Red USB 3.0 for PC Laptop and Mac, 2 Months Adobe CC Photography (STDR5000103)
----------------------------------------------------------------------------------------------------
CATEGORY:
Computers

FEATURES:
['Want an 

In [13]:
def build_compact_target_info(row):
    parts = []

    category = clean_metadata_value(row["main_category"])
    if category:
        parts.append(f"Category: {category}")

    features = clean_metadata_value(row["features"])
    if features:
        parts.append(f"Product information: {features}")

    categories = clean_metadata_value(row["categories"])
    if categories:
        parts.append(f"Category path: {categories}")

    return "\n".join(parts)


dataset["target_raw_info"] = dataset.apply(build_compact_target_info, axis=1)

print("Compact target information created.")
print("\nExample:")
print(dataset.loc[1, "target_raw_info"])

print("\nAverage length:",
      dataset["target_raw_info"].str.len().mean())

print("Maximum length:",
      dataset["target_raw_info"].str.len().max())

print("Missing:",
      (dataset["target_raw_info"].str.strip() == "").sum())

Compact target information created.

Example:
Category: Computers
Product information: ['Want an easy, on-the-go USB drive for storage? Seagate Backup Plus Portable hard drive offers 5TB', 'The perfect complimentary laptop hard drive featuring a minimalist metal enclosure', 'Simply plug this external hard drive for Mac and Windows into a computer via the included USB 3.0 cable to back up files with a single click or schedule automatic daily, weekly, or monthly backups. Reformatting may be required for use with Time Machine.', 'Take advantage of a complimentary two-month membership to the Adobe Creative Cloud Photography Plan for access to awesome photo and video editing apps', 'Enjoy long-term peace of mind with the included two-year limited warranty']
Category path: ['Electronics', 'Computers & Accessories', 'Data Storage', 'External Hard Drives']

Average length: 975.9447742399713
Maximum length: 5004
Missing: 0


## 3. Qwen Model Configuration

Qwen2.5-7B-Instruct is used to generate a concise synthetic shopping
query from a user's historical preference profile and a held-out target
product.

The user preference profile represents information available from the
user's past interactions, while the target product represents the
held-out future purchase used to construct the synthetic query.

The model is instructed to express the general shopping need and relevant
product characteristics without revealing the exact identity of the
target product.

In [14]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
tokenizer= AutoTokenizer.from_pretrained(MODEL_NAME,local_files_only=True)
model= AutoModelForCausalLM.from_pretrained(MODEL_NAME,local_files_only=True,torch_dtype=torch.float16,device_map="auto")
model.eval()
print("Qwen loaded successfully.")
print("Model device:", next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████████████████████████████████████| 339/339 [00:04<00:00, 70.46it/s]


Qwen loaded successfully.
Model device: cuda:0


## 4. Synthetic Query Generation Prompt

The system prompt defines the requirements for target-aware synthetic
shopping-query generation.

The generated query should:

- be natural and concise;
- reflect the user's demonstrated preferences where relevant;
- correspond to the general product type and useful characteristics of
  the held-out target;
- avoid revealing the exact identity of the target product;
- avoid copying the target product title;
- avoid unsupported requirements or use cases;
- represent a plausible pre-purchase shopping query.

In [15]:
SYSTEM_PROMPT = """
You are generating a realistic synthetic shopping query for a research dataset.

You will be given:
1. A user's purchase preference profile — describing their purchase history,
   preferred categories, and demonstrated product preferences.
2. A target product — the item representing the user's held-out future purchase.

Your task is to generate ONE natural shopping query that:
- Expresses the user's need for the TARGET PRODUCT TYPE and relevant
  functional characteristics.
- Is grounded in the user's profile where relevant (e.g. compatible
  devices, preferred brands, or use cases evident from history).
- Does NOT copy or name the target product.
- Does NOT include the target product's exact brand, model number, ASIN,
  exact title, or other unique identifying information.
- May express general characteristics of the target product such as
  product type, features, specifications, compatibility, or functionality.
- Reads like a realistic search query a real user would type before
  purchasing the target product.

Requirements:
- The query should be inferable from the user's profile and the general
  characteristics of the target product.
- Express the functional need, relevant product characteristics,
  compatibility requirement, or intended use rather than the product's
  exact identity.
- Keep the query concise and natural, like a real search or shopping
  request (typically one sentence).
- Do not invent needs, requirements, or use cases that are unsupported by
  either the user's profile or the target product information.
- Do not reveal that the query was generated using the target product.
- Do not include reasoning or explanation.

Return ONLY valid JSON using exactly this schema:

{
  "query": "..."
}

Do not use markdown or code fences.
"""

## 5. Single-Profile Generation Test

Before generating the complete dataset, query generation is tested on an
individual preference profile to verify that the model, tokenizer, prompt,
and JSON output format are functioning correctly.

In [16]:
sample_profile = dataset.loc[0, "profile_text"]
sample_target = dataset.loc[0, "target_raw_info"]

user_prompt = f"""
User Preference Profile:
{sample_profile}

Target Product Information:
{sample_target}
""".strip()

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.6,
        top_p=0.9,
        do_sample=True
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(response)

{
  "query": "Looking for a comfortable over-ear headphone with good bass for outdoor use and compatible with my iPhone"
}


In [17]:
def parse_query_response(response):
    """
    Extract the query from Qwen's response.
    Returns None if a valid query cannot be extracted.
    """

    if not isinstance(response, str):
        return None
    response= response.strip()
    #Remove accidental markdown code fences
    response= re.sub(r"```json\s*", "", response, flags=re.IGNORECASE)
    response= re.sub(r"```\s*", "", response)
    #Try to parse the complete response as JSON
    try:
        data= json.loads(response)
        if isinstance(data, dict):
            query= data.get("query")
            if isinstance(query, str):
                query= query.strip()
                if query:
                    return query
    except json.JSONDecodeError:
        pass
    #Fallback: extract the query field from malformed JSON
    match= re.search(r'"query"\s*:\s*"([^"]+)"',response,flags=re.IGNORECASE)
    if match:
        query= match.group(1).strip()
        if query:
            return query
    return None

In [18]:
test_response = """
{
  "query": "looking for a high speed micro sd card for my action camera"
}
"""
print(parse_query_response(test_response))

looking for a high speed micro sd card for my action camera


## 7. Query Generation Function

The generation function applies the system prompt to an individual user
preference profile and returns a single synthetic shopping query.

Sampling parameters are used to provide variation across users while
maintaining relatively controlled and concise outputs.

In [19]:
def generate_query(profile_text, target_product_info):
    """
    Generate one synthetic shopping query from a user preference profile
    and held-out target product information.
    Returns the query string or None if generation/parsing fails.
    """
    user_prompt = f"""
    User Preference Profile:
    {profile_text}

    Target Product Information:
    {target_product_info}
    """.strip()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.6,
            top_p=0.9,
            do_sample=True
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return parse_query_response(response)

## 8. Generation Reliability Test

A small sample of 20 user profiles is used to verify that the generation
function consistently produces valid queries before running the complete
dataset.

In [20]:
test_results = []

for idx in dataset.index[:20]:

    profile = dataset.loc[idx, "profile_text"]
    target = dataset.loc[idx, "target_raw_info"]

    query = generate_query(profile, target)

    test_results.append({
        "dataset_index": idx,
        "sample_id": dataset.loc[idx, "sample_id"],
        "query": query
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,dataset_index,sample_id,query
0,0,0,looking for comfortable over-ear headphones wi...
1,1,1,looking for a reliable external hard drive for...
2,2,2,looking for portable on-ear headphones with go...
3,3,3,looking for good quality on-ear headphones wit...
4,4,4,Looking for a long-lasting replacement battery...
5,5,5,looking for a car backup camera system with ma...
6,6,6,looking for a high-quality TV antenna for my R...
7,7,7,looking for a battery holder for my hobby proj...
8,8,8,Looking for a battery holder that fits my Ardu...
9,9,9,looking for a reliable laptop charger with mul...


In [21]:
print(
    "Successful:",
    test_results_df["query"].notna().sum(),
    "/",
    len(test_results_df)
)

Successful: 20 / 20


## 9. Full Synthetic Query Generation

The complete set of user preference profiles is processed using Qwen.

Checkpointing is used to save generated queries periodically. This allows
the process to resume from the last completed checkpoint if the SLURM job
or GPU session is interrupted.

In [22]:
import time

CHECKPOINT_PATH= GENERATED_DIR/"synthetic_queries_checkpoint.parquet"
BATCH_SIZE= 50

#Load existing checkpoint if available
if CHECKPOINT_PATH.exists():
    checkpoint_df= pd.read_parquet(CHECKPOINT_PATH)
    results= checkpoint_df.to_dict("records")
    completed_indices= set(checkpoint_df["dataset_index"].astype(int))
    print(f"Resuming from checkpoint: {len(results)} completed")
else:
    results= []
    completed_indices= set()
    print("Starting generation from scratch.")
print(f"Total profiles: {len(dataset)}")
print(f"Remaining profiles: {len(dataset) - len(completed_indices)}")

Resuming from checkpoint: 2550 completed
Total profiles: 5559
Remaining profiles: 3009


In [23]:
if CHECKPOINT_PATH.exists():
    checkpoint_df= pd.read_parquet(CHECKPOINT_PATH)
else:
    checkpoint_df= pd.DataFrame(columns=["dataset_index", "sample_id", "query"])
completed_indices= set(checkpoint_df["dataset_index"].astype(int).tolist())
results= checkpoint_df.to_dict("records")
print(f"Already completed: {len(completed_indices)}")
print(f"Remaining: {len(dataset) - len(completed_indices)}")

Already completed: 2550
Remaining: 3009


In [28]:
CHECKPOINT_PATH.unlink(missing_ok=True)

results = []
completed_indices = set()

print("Old checkpoint removed.")
print(f"Profiles to generate: {len(dataset)}")

Old checkpoint removed.
Profiles to generate: 5559


In [24]:
CHECKPOINT_PATH = GENERATED_DIR / "synthetic_queries_checkpoint.parquet"

checkpoint_df = pd.read_parquet(CHECKPOINT_PATH)

completed_indices = set(
    checkpoint_df["dataset_index"].astype(int)
)

results = checkpoint_df.to_dict("records")

print(f"Checkpoint rows: {len(checkpoint_df)}")
print(f"Completed: {len(completed_indices)}")
print(f"Remaining: {len(dataset) - len(completed_indices)}")

Checkpoint rows: 2550
Completed: 2550
Remaining: 3009


In [25]:
start_time= time.time()
new_since_checkpoint= 0
for idx in dataset.index:
    if idx in completed_indices:
        continue
    profile= dataset.loc[idx, "profile_text"]
    target= dataset.loc[idx, "target_raw_info"]
    try:
        query= generate_query(profile, target)
        # Retry once if generation/parsing failed
        if query is None:
            print(f"Retrying index {idx}...")
            query= generate_query(profile, target)
        results.append({
            "dataset_index": int(idx),
            "sample_id": dataset.loc[idx, "sample_id"],
            "user_id": dataset.loc[idx, "user_id"],
            "query": query
        })
        completed_indices.add(idx)
        new_since_checkpoint += 1
    except Exception as e:
        print(f"ERROR at index {idx}: {e}")
        results.append({
            "dataset_index": int(idx),
            "sample_id": dataset.loc[idx, "sample_id"],
            "user_id": dataset.loc[idx, "user_id"],
            "query": None
        })
        completed_indices.add(idx)
        new_since_checkpoint += 1
    #Save checkpoint every 50 new profiles
    if new_since_checkpoint >= BATCH_SIZE:
        checkpoint_df= pd.DataFrame(results)
        checkpoint_df.to_parquet(CHECKPOINT_PATH,index=False)
        elapsed= time.time()-start_time
        completed= len(completed_indices)
        remaining= len(dataset)-completed
        print(
            f"Checkpoint: {completed}/{len(dataset)} "
            f"({completed / len(dataset) * 100:.1f}%) | "
            f"Remaining: {remaining} | "
            f"Elapsed: {elapsed / 60:.1f} min"
        )
        new_since_checkpoint= 0
#Final save
checkpoint_df= pd.DataFrame(results)
checkpoint_df.to_parquet(CHECKPOINT_PATH,index=False)
elapsed= time.time()-start_time
print("\nGeneration complete!")
print(f"Total generated: {len(checkpoint_df)}")
print(f"Total time: {elapsed / 3600:.2f} hours")

Checkpoint: 2600/5559 (46.8%) | Remaining: 2959 | Elapsed: 1.0 min
Checkpoint: 2650/5559 (47.7%) | Remaining: 2909 | Elapsed: 2.1 min
Checkpoint: 2700/5559 (48.6%) | Remaining: 2859 | Elapsed: 3.3 min
Checkpoint: 2750/5559 (49.5%) | Remaining: 2809 | Elapsed: 4.5 min
Checkpoint: 2800/5559 (50.4%) | Remaining: 2759 | Elapsed: 5.6 min
Checkpoint: 2850/5559 (51.3%) | Remaining: 2709 | Elapsed: 6.6 min
Checkpoint: 2900/5559 (52.2%) | Remaining: 2659 | Elapsed: 7.7 min
Checkpoint: 2950/5559 (53.1%) | Remaining: 2609 | Elapsed: 8.9 min
Checkpoint: 3000/5559 (54.0%) | Remaining: 2559 | Elapsed: 9.9 min
Checkpoint: 3050/5559 (54.9%) | Remaining: 2509 | Elapsed: 10.9 min
Checkpoint: 3100/5559 (55.8%) | Remaining: 2459 | Elapsed: 11.9 min
Checkpoint: 3150/5559 (56.7%) | Remaining: 2409 | Elapsed: 13.0 min
Checkpoint: 3200/5559 (57.6%) | Remaining: 2359 | Elapsed: 14.0 min
Checkpoint: 3250/5559 (58.5%) | Remaining: 2309 | Elapsed: 15.1 min
Checkpoint: 3300/5559 (59.4%) | Remaining: 2259 | Elapsed

In [26]:
CHECKPOINT_PATH= GENERATED_DIR/"synthetic_queries_checkpoint.parquet"
generated= pd.read_parquet(CHECKPOINT_PATH)
print("Generated rows:", len(generated))
print("Columns:", generated.columns.tolist())

Generated rows: 5559
Columns: ['dataset_index', 'sample_id', 'user_id', 'query']


## 10. Synthetic Query Dataset Quality Control

The generated dataset is evaluated for:

- missing queries;
- empty queries;
- duplicate queries;
- query-length distribution;
- and profile-query consistency.

Duplicate queries are retained because different users may have similar
preference profiles and therefore legitimately produce the same shopping
intent.

In [27]:
print("Missing queries:", generated["query"].isna().sum())
print("Missing percentage:",generated["query"].isna().mean()*100,"%")

Missing queries: 0
Missing percentage: 0.0 %


In [28]:
empty_queries= (generated["query"].fillna("").str.strip().eq(""))
print("Empty queries:", empty_queries.sum())

Empty queries: 0


In [29]:
print("Unique queries:", generated["query"].nunique())
print("Duplicate query rows:", generated.duplicated("query").sum())

Unique queries: 5465
Duplicate query rows: 94


In [31]:
duplicate_counts= (generated["query"].value_counts().loc[lambda x: x > 1])
print("Number of duplicated query strings:", len(duplicate_counts))
print("Total duplicate rows:", duplicate_counts.sum() - len(duplicate_counts))
display(duplicate_counts.head(20))

Number of duplicated query strings: 70
Total duplicate rows: 94


query
looking for wireless earbuds with good sound quality and long battery life for workouts                         7
Looking for wireless earbuds with good sound quality and long battery life for workouts                         5
looking for wireless earbuds with good noise cancellation and long battery life for workouts                    5
looking for a smart speaker to control my home devices and play music                                           4
looking for wireless earbuds with good sound quality and sweat resistance for workouts                          4
looking for a portable bluetooth speaker with good sound quality and long battery life for outdoor use          4
looking for a protective case for my tablet                                                                     3
looking for a fire tv stick with alexa voice remote for streaming movies and tv shows                           3
looking for a protective case for my new tablet                                   

In [32]:
generated["query_length"]= (generated["query"].fillna("").str.split().str.len())
print(generated["query_length"].describe())

count    5559.000000
mean       14.395215
std         3.088963
min         4.000000
25%        12.000000
50%        14.000000
75%        16.000000
max        30.000000
Name: query_length, dtype: float64


In [33]:
display(generated[["sample_id", "query"]].sample(30, random_state=42))

,sample_id,query
4963,4963,looking for a cable to charge and sync my tabl...
4943,4943,looking for a slim full hd monitor with great ...
4345,4345,Looking for a portable Bluetooth speaker with ...
230,230,looking for a waterproof housing for my go pro...
1044,1044,Looking for noise-cancelling earbuds compatibl...
3988,3988,looking for a high-quality screen protector fo...
3121,3121,looking for a reliable cable modem to boost my...
2662,2662,Looking for over-ear headphones with good soun...
3259,3259,looking for a premium case for my samsung gala...
381,381,looking for a quiet keyboard with a large touc...


## 12. Profile–Query Grounding Evaluation

A sample of generated profile-query pairs is inspected to determine whether
the generated shopping intent is supported by the corresponding user
preference profile.

The evaluation distinguishes between:

- Fully grounded: the query is directly supported by the profile.
- Partially grounded: the main product interest is supported, but the query
  introduces an unsupported attribute or constraint.
- Unsupported: the generated query has little or no evidence in the profile.

In [34]:
qc= generated.merge(dataset[["sample_id", "user_id", "profile_text"]],on=["sample_id", "user_id"],how="left")
print("QC shape:", qc.shape)
grounding_sample= (qc[["sample_id", "profile_text", "query"]].sample(100, random_state=42).reset_index(drop=True))
display(grounding_sample.head(10))

QC shape: (5559, 6)


,sample_id,profile_text,query
0,4963,User Preference Profile\n\nThe user has primar...,looking for a cable to charge and sync my tabl...
1,4943,User Preference Profile\n\nThe user has primar...,looking for a slim full hd monitor with great ...
2,4345,User Preference Profile\n\nThe user has primar...,Looking for a portable Bluetooth speaker with ...
3,230,User Preference Profile\n\nThe user has primar...,looking for a waterproof housing for my go pro...
4,1044,User Preference Profile\n\nThe user has primar...,Looking for noise-cancelling earbuds compatibl...
5,3988,User Preference Profile\n\nThe user has primar...,looking for a high-quality screen protector fo...
6,3121,User Preference Profile\n\nThe user has primar...,looking for a reliable cable modem to boost my...
7,2662,User Preference Profile\n\nThe user has primar...,Looking for over-ear headphones with good soun...
8,3259,User Preference Profile\n\nThe user has primar...,looking for a premium case for my samsung gala...
9,381,User Preference Profile\n\nThe user has primar...,looking for a quiet keyboard with a large touc...


In [35]:
print("Rows:", len(qc))
print("Missing profiles:", qc["profile_text"].isna().sum())
print("Missing queries:", qc["query"].isna().sum())
print("Duplicate sample IDs:", qc["sample_id"].duplicated().sum())

Rows: 5559
Missing profiles: 0
Missing queries: 0
Duplicate sample IDs: 0


## 13. Create Final Synthetic Query Dataset

Following quality-control checks, the generated queries are merged with
their corresponding user identifiers and preference profiles.

The final dataset retains the user-level records and generated query while
excluding temporary generation fields used only for checkpointing.

In [36]:
final_queries= qc[["sample_id","user_id","profile_text","query"]].copy()
final_queries= final_queries.rename(columns={"query": "synthetic_query"})
FINAL_PATH= (PROJECT_ROOT/"data"/"generated"/"synthetic_queries.parquet")
final_queries.to_parquet(FINAL_PATH,index=False)
print("Final dataset saved to:")
print(FINAL_PATH)
print("\nFinal shape:")
print(final_queries.shape)
display(final_queries.head())

Final dataset saved to:
/users/adhp511/query-conditioned-recommender/data/generated/synthetic_queries.parquet

Final shape:
(5559, 4)


,sample_id,user_id,profile_text,synthetic_query
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,looking for comfortable over-ear headphones wi...
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,looking for a reliable external hard drive for...
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,looking for good quality on-ear headphones com...
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,looking for good quality on-ear headphones wit...
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,Looking for long-lasting replacement batteries...


In [37]:
print(FINAL_PATH.exists())
print(FINAL_PATH.stat().st_size/(1024**2), "MB")

True
1.4267005920410156 MB
